# 01-04 Softmax 函数公式推导

Softmax 常用于多分类任务的输出层。它的核心作用是：把一组任意实数分数转换成一组概率，并且所有概率之和为 $1$。

## 1. 为什么需要 Softmax

多分类模型通常先输出 $K$ 个原始分数：

$$
\mathbf{z}=[z_1,z_2,\dots,z_K]
$$

这些分数也叫 logits。它们不是概率，因为可能为负，也不一定相加为 $1$。

我们希望得到一组概率：

$$
\mathbf{p}=[p_1,p_2,\dots,p_K]
$$

并满足：

$$
0<p_i<1
$$

$$
\sum_{i=1}^{K}p_i=1
$$

Softmax 就是把 logits 转成概率分布的方法。

### 1.1 具体计算示例

假设一个三分类问题（猫、狗、鸟），模型输出的原始 logits 是：

$$
\mathbf{z}=[2.0, 1.0, 0.5]
$$

这三个数就是“原始分数”。$2.0$ 最高说明模型更倾向第一类（猫），但直接看它们并不直观——加起来不是 $1$，也不代表概率。

下面一步一步用 Softmax 把它转成概率。

**第 1 步：对每个 logit 取指数**

$$
e^{z_1}=e^{2.0}\approx 7.389
$$

$$
e^{z_2}=e^{1.0}\approx 2.718
$$

$$
e^{z_3}=e^{0.5}\approx 1.649
$$

取指数的目的是把负数变正数，同时放大差距（$2.0$ 和 $1.0$ 差 $1$ 倍，指数后 $7.389$ 和 $2.718$ 差约 $2.7$ 倍）。

**第 2 步：求分母（所有指数的总和）**

$$
\sum_{j=1}^{3}e^{z_j}=7.389+2.718+1.649=11.756
$$

这个总和是归一化的基准，确保最后所有概率加起来等于 $1$。

**第 3 步：每个指数除以总和**

$$
p_1=\frac{7.389}{11.756}\approx 0.629
$$

$$
p_2=\frac{2.718}{11.756}\approx 0.231
$$

$$
p_3=\frac{1.649}{11.756}\approx 0.140
$$

**结果**

$$
\mathbf{p}=[0.629, 0.231, 0.140]
$$

验证：$0.629+0.231+0.140=1.000$，每个值都在 $(0,1)$ 之间。

现在可以直观解读：模型认为这张图片有 **62.9%** 的概率是猫，**23.1%** 是狗，**14.0%** 是鸟。这就是把“原始分数”转化为“概率分布”的过程。

### 1.2 直观总结

| 阶段 | 数值 | 特点 |
|------|------|------|
| 原始 logits | `[2.0, 1.0, 0.5]` | 可正可负，和≠1，不能当概率 |
| 取指数 $e^{z_i}$ | `[7.389, 2.718, 1.649]` | 全正数，放大了类别差异 |
| 除以总和（归一化） | `[0.629, 0.231, 0.140]` | 和为1，就是概率分布 |

一句话概括：**指数保证为正，除以总和保证和为1。** 这两步就把没有约束的原始分数变成了合法的概率分布。

## 2. 从二分类 Sigmoid 到多分类 Softmax

二分类时，我们可以用 Sigmoid 输出类别 $1$ 的概率：

$$
p=\frac{1}{1+e^{-z}}
$$

也可以把二分类看成两个类别分数 $z_0$ 和 $z_1$ 的竞争：

$$
p_1=\frac{e^{z_1}}{e^{z_0}+e^{z_1}}
$$

如果令 $z=z_1-z_0$，那么：

$$
p_1=\frac{e^{z_1}}{e^{z_0}+e^{z_1}}
$$

分子分母同时除以 $e^{z_1}$：

$$
p_1=\frac{1}{e^{z_0-z_1}+1}
$$

$$
p_1=\frac{1}{1+e^{-(z_1-z_0)}}
$$

$$
p_1=\sigma(z_1-z_0)
$$

所以 Softmax 可以看成 Sigmoid 在多分类场景下的推广。

## 3. Softmax 公式怎么来

我们需要把每个类别分数 $z_i$ 变成正数。指数函数正好满足：

$$
e^{z_i}>0
$$

先得到每个类别的非归一化权重：

$$
s_i=e^{z_i}
$$

为了让所有类别的概率加起来等于 $1$，用所有权重的总和进行归一化：

$$
p_i=\frac{s_i}{\sum_{j=1}^{K}s_j}
$$

代入 $s_i=e^{z_i}$：

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

这就是 Softmax：

$$
\operatorname{softmax}(z_i)=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

## 4. 为什么输出一定是概率分布

首先，因为指数函数恒正：

$$
e^{z_i}>0
$$

所以：

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}>0
$$

其次，所有概率求和：

$$
\sum_{i=1}^{K}p_i=\sum_{i=1}^{K}\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

因为分母和 $i$ 无关，可以提出：

$$
\sum_{i=1}^{K}p_i=\frac{\sum_{i=1}^{K}e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

分子和分母是同一个总和，所以：

$$
\sum_{i=1}^{K}p_i=1
$$

因此 Softmax 的输出就是合法的概率分布。

## 5. 平移不变性与数值稳定

Softmax 有一个非常重要的性质：给所有 logits 同时加上同一个常数 $c$，结果不变。

$$
\operatorname{softmax}(z_i+c)=\frac{e^{z_i+c}}{\sum_{j=1}^{K}e^{z_j+c}}
$$

因为：

$$
e^{z_i+c}=e^c e^{z_i}
$$

所以：

$$
\operatorname{softmax}(z_i+c)=\frac{e^c e^{z_i}}{\sum_{j=1}^{K}e^c e^{z_j}}
$$

上下约去 $e^c$：

$$
\operatorname{softmax}(z_i+c)=\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

实际计算时，常取：

$$
c=-\max(\mathbf{z})
$$

这样可以避免 $e^{z_i}$ 因为 $z_i$ 太大而溢出。稳定版 Softmax 写成：

$$
p_i=\frac{e^{z_i-\max(\mathbf{z})}}{\sum_{j=1}^{K}e^{z_j-\max(\mathbf{z})}}
$$

## 6. Softmax 的优缺点、产生原因与应用场景

### 6.1 优点

第一个优点是输出可以解释成多分类概率。Softmax 的输出满足：

$$
0<p_i<1
$$

$$
\sum_{i=1}^{K}p_i=1
$$

所以它非常适合表达“样本属于每个类别的概率”。

第二个优点是会放大类别之间的相对差异。因为 Softmax 使用指数函数：

$$
e^{z_i}
$$

如果某个类别的 logit 更大，指数运算会进一步放大它和其他类别的差距，从而让概率更集中到高分类别上。

第三个优点是和交叉熵损失配合后梯度形式非常简洁：

$$
\frac{\partial \mathcal{L}}{\partial z_i}=p_i-y_i
$$

这个结果让多分类模型的优化非常直接：预测概率和真实标签差多少，就按这个差距去更新 logit。

### 6.2 缺点

第一个缺点是容易出现数值溢出。因为 Softmax 使用指数函数，如果 $z_i$ 很大：

$$
e^{z_i}\to +\infty
$$

计算机中可能发生溢出。因此实际计算时通常使用稳定版：

$$
p_i=\frac{e^{z_i-\max(\mathbf{z})}}{\sum_{j=1}^{K}e^{z_j-\max(\mathbf{z})}}
$$

第二个缺点是类别之间互相竞争。Softmax 的分母包含所有类别：

$$
\sum_{j=1}^{K}e^{z_j}
$$

所以提高某个类别的概率，通常会降低其他类别的概率。这适合单标签多分类，但不适合多标签分类。

例如一张图片既可以有“猫”，也可以有“沙发”，这时多个标签可以同时为真，使用 Softmax 就不合适，更常见的是对每个标签分别使用 Sigmoid。

第三个缺点是它不适合作为隐藏层激活函数。Softmax 会把一组神经元输出强制归一化成概率分布：

$$
\sum_{i=1}^{K}p_i=1
$$

隐藏层通常需要自由表达多个特征，而不是让这些特征互相竞争成概率，所以 Softmax 一般放在输出层，而不是隐藏层。

### 6.3 这些优缺点为什么会产生

Softmax 的概率解释来自两个步骤。第一步，用指数函数保证每个类别权重为正：

$$
s_i=e^{z_i}>0
$$

第二步，用总和归一化：

$$
p_i=\frac{s_i}{\sum_{j=1}^{K}s_j}
$$

正是这个“变正 + 归一化”的结构，让 Softmax 输出成为概率分布。

类别竞争也来自同一个分母。因为每个类别概率都共享：

$$
\sum_{j=1}^{K}e^{z_j}
$$

所以某个类别的 logit 变化，会影响所有类别的概率。这就是 Softmax 适合单标签多分类、不适合多标签分类的根本原因。

数值溢出则来自指数函数增长太快。比如当 $z$ 很大时，$e^z$ 会远远超出普通浮点数能稳定表示的范围。因此实际实现一定要做平移：

$$
\mathbf{z}\leftarrow \mathbf{z}-\max(\mathbf{z})
$$

### 6.4 应用场景

第一个场景是单标签多分类任务。例如 MNIST 手写数字分类中，每张图片只属于 $0$ 到 $9$ 中的一个类别。模型输出 $10$ 个 logits：

$$
\mathbf{z}=[z_0,z_1,\dots,z_9]
$$

再用 Softmax 得到每个数字的概率：

$$
\mathbf{p}=\operatorname{softmax}(\mathbf{z})
$$

第二个场景是神经网络的多分类输出层。输出层常写成：

$$
\mathbf{p}=\operatorname{softmax}(\mathbf{W}\mathbf{h}+\mathbf{b})
$$

其中 $\mathbf{h}$ 是最后一个隐藏层的输出。

第三个场景是注意力机制中的权重归一化。Attention 中常用 Softmax 把相关性分数变成注意力权重：

$$
\operatorname{Attention}(\mathbf{Q},\mathbf{K},\mathbf{V})=\operatorname{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^{T}}{\sqrt{d_k}}\right)\mathbf{V}
$$

这里 Softmax 的作用不是分类，而是把一组匹配分数转换成权重分布。

实践中还要记住：在 PyTorch 多分类训练中，通常直接使用 `nn.CrossEntropyLoss()`，模型输出 logits 即可，不需要手动先加 Softmax。

## 7. 和交叉熵损失的关系

多分类任务中，Softmax 常和交叉熵损失一起使用。

如果真实标签是 one-hot 向量：

$$
\mathbf{y}=[y_1,y_2,\dots,y_K]
$$

预测概率是：

$$
\mathbf{p}=\operatorname{softmax}(\mathbf{z})
$$

交叉熵损失是：

$$
\mathcal{L}=-\sum_{i=1}^{K}y_i\log p_i
$$

如果真实类别是第 $t$ 类，那么只有 $y_t=1$，其他 $y_i=0$，因此：

$$
\mathcal{L}=-\log p_t
$$

这表示：真实类别的预测概率越大，损失越小。

## 8. Softmax + 交叉熵的梯度结论

Softmax 和交叉熵组合后，有一个非常简洁的梯度结果：

$$
\frac{\partial \mathcal{L}}{\partial z_i}=p_i-y_i
$$

这个结果很重要。它说明每个类别 logit 的更新方向，就是预测概率和真实标签之间的差距。

如果某个错误类别的概率 $p_i$ 太大，而 $y_i=0$，那么：

$$
p_i-y_i>0
$$

梯度下降会压低这个类别的 logit。

如果真实类别 $t$ 的概率 $p_t$ 太小，而 $y_t=1$，那么：

$$
p_t-y_t<0
$$

梯度下降会抬高真实类别的 logit。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(logits):
    shifted = logits - np.max(logits)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits)

logits = np.array([1.2, 3.5, 0.8])
probs = softmax(logits)
target = np.array([0, 1, 0])
gradient = probs - target

print('logits:', logits)
print('softmax probabilities:', probs.round(4))
print('gradient p - y:', gradient.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
classes = ['class 1', 'class 2', 'class 3']

axes[0].bar(classes, probs, color=['#93C5FD', '#FCA5A5', '#86EFAC'])
axes[0].set_ylim(0, 1)
axes[0].set_title('Softmax Probabilities')
for i, p in enumerate(probs):
    axes[0].text(i, p + 0.03, f'{p:.2f}', ha='center')

axes[1].bar(classes, gradient, color=['#93C5FD', '#FCA5A5', '#86EFAC'])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Gradient: p - y')

plt.tight_layout()
plt.show()


## 9. 在 PyTorch 中的注意点

在 PyTorch 多分类训练中，通常使用：

```python
criterion = nn.CrossEntropyLoss()
```

`nn.CrossEntropyLoss()` 内部已经包含了 `log_softmax` 和负对数似然损失，所以模型最后一层通常直接输出 logits，不需要手动加 Softmax。

训练时：

$$
\text{model output}=\mathbf{z}
$$

推理时，如果需要查看概率，再单独计算：

$$
\mathbf{p}=\operatorname{softmax}(\mathbf{z})
$$